In [16]:
from collections.abc import Callable
from pathlib import Path
from typing import Any, Literal

import equinox as eqx
import jax
import xarray as xr
from context_flux_no.simulations.pde import CubicFlux1D
from jaxtyping import Array, PRNGKeyArray
from tqdm import tqdm


jax.config.update("jax_enable_x64", True)


In [2]:
def sample_pde(
    pde_factory: Callable[[Any], eqx.Module],
    *,
    seed: int = 0,
    **kwargs: dict[str, tuple[float, float]],
) -> eqx.Module:
    keys = jax.random.split(jax.random.key(seed), len(kwargs))

    param_dict = dict()
    for rng_key, (param_name, sample_range) in zip(keys, kwargs.items()):
        param_dict[param_name] = float(
            jax.random.uniform(
                rng_key,
                minval=sample_range[0],
                maxval=sample_range[1],
            )
        )

    return pde_factory(**param_dict)

In [3]:
pde = sample_pde(CubicFlux1D, a=(-1.0, 1.0), b=(-1.0, 1.0), c=(-1.0, 1.0))
pde

CubicFlux1D(a=0.9398959959797777, b=-0.8387512666713679, c=0.30384451116312405)

In [5]:
from context_flux_no.waveforms.grf import GaussianCov, GaussianRandomField1D


u0_func = lambda x_: GaussianRandomField1D(GaussianCov(0.05)).sample(
    x_, jax.random.key(0)
)

solution = pde.solve_pyclaw(u0_func, (-1.0, 1.0), 256, (0.0, 0.1), 100, "periodic")
solution

2025-10-30 12:13:05,320 INFO CLAW: Solution 0 computed for time t=0.000000
2025-10-30 12:13:05,324 INFO CLAW: Solution 1 computed for time t=0.001000
2025-10-30 12:13:05,327 INFO CLAW: Solution 2 computed for time t=0.002000
2025-10-30 12:13:05,331 INFO CLAW: Solution 3 computed for time t=0.003000
2025-10-30 12:13:05,333 INFO CLAW: Solution 4 computed for time t=0.004000
2025-10-30 12:13:05,338 INFO CLAW: Solution 5 computed for time t=0.005000
2025-10-30 12:13:05,344 INFO CLAW: Solution 6 computed for time t=0.006000
2025-10-30 12:13:05,348 INFO CLAW: Solution 7 computed for time t=0.007000
2025-10-30 12:13:05,352 INFO CLAW: Solution 8 computed for time t=0.008000
2025-10-30 12:13:05,354 INFO CLAW: Solution 9 computed for time t=0.009000
2025-10-30 12:13:05,357 INFO CLAW: Solution 10 computed for time t=0.010000
2025-10-30 12:13:05,361 INFO CLAW: Solution 11 computed for time t=0.011000
2025-10-30 12:13:05,365 INFO CLAW: Solution 12 computed for time t=0.012000
2025-10-30 12:13:05,36

<xarray.Dataset> Size: 210kB
Dimensions:  (sample: 1, t: 101, dim: 1, x: 256, param: 3)
Coordinates:
  * t        (t) float64 808B 0.0 0.001 0.002 0.003 ... 0.097 0.098 0.099 0.1
  * x        (x) float64 2kB -0.9961 -0.9883 -0.9805 ... 0.9805 0.9883 0.9961
  * dim      (dim) <U1 4B 'u'
  * param    (param) <U1 12B 'a' 'b' 'c'
Dimensions without coordinates: sample
Data variables:
    values   (sample, t, dim, x) float64 207kB -0.2243 -0.3045 ... -0.6573
    coeffs   (sample, param) float64 24B 0.9399 -0.8388 0.3038

In [6]:
def sample_coefficients_uniform(
    key: PRNGKeyArray, coeff_range_dict: dict[str, tuple[float, float]]
):
    subkeys = jax.random.split(key, len(coeff_range_dict))
    return {
        name: float(
            jax.random.uniform(subkey, minval=coeff_range[0], maxval=coeff_range[1])
        )
        for subkey, (name, coeff_range) in zip(subkeys, coeff_range_dict.items())
    }


In [13]:
def generate_dataset(
    n_samples: int,
    pde_factory: Callable[[Any], eqx.Module],
    initial_condition_fn: Callable[[Array, PRNGKeyArray], Array],
    coeff_range_dict: dict,
    x_span: tuple[float, float],
    Nx: int,
    t_span: tuple[float, float],
    Nt: int,
    bc: Literal["periodic"] = "periodic",
    seed: int = 0,
):
    keys = jax.random.split(jax.random.key(seed), n_samples)

    solutions = []
    for key in tqdm(keys):
        key_coeff, key_ic = jax.random.split(key)
        coeffs = sample_coefficients_uniform(key_coeff, coeff_range_dict)
        pde = pde_factory(**coeffs)

        sol = pde.solve_pyclaw(
            lambda u0: initial_condition_fn(u0, key_ic),
            x_span,
            Nx,
            t_span,
            Nt,
            bc,
            verbose=False,
        )
        solutions.append(sol)

    return xr.concat(solutions, "sample")

In [14]:
dataset = generate_dataset(
    n_samples=100,
    pde_factory=CubicFlux1D,
    initial_condition_fn=GaussianRandomField1D(GaussianCov(0.05)).sample,
    coeff_range_dict={"a": (-1.0, 1.0), "b": (-1.0, 1.0), "c": (-1.0, 1.0)},
    x_span=(-1, 1),
    Nx=256,
    t_span=(0, 0.1),
    Nt=100,
    seed=0,
)

100%|██████████| 100/100 [00:07<00:00, 13.39it/s]


In [15]:
dataset

<xarray.Dataset> Size: 21MB
Dimensions:  (sample: 100, t: 101, dim: 1, x: 256, param: 3)
Coordinates:
  * t        (t) float64 808B 0.0 0.001 0.002 0.003 ... 0.097 0.098 0.099 0.1
  * x        (x) float64 2kB -0.9961 -0.9883 -0.9805 ... 0.9805 0.9883 0.9961
  * dim      (dim) <U1 4B 'u'
  * param    (param) <U1 12B 'a' 'b' 'c'
Dimensions without coordinates: sample
Data variables:
    values   (sample, t, dim, x) float64 21MB 0.7198 0.3809 ... -0.1633
    coeffs   (sample, param) float64 2kB -0.4814 -0.2967 ... -0.1377 0.806

In [17]:
savedir = Path("../../data/")
savedir.mkdir(parents=True, exist_ok=True)
dataset.to_netcdf(savedir / "cubic_no_source_train.hdf5")